In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import numpy as np
from tqdm import tqdm
import warnings
from PIL import Image, ImageFile
warnings.filterwarnings('ignore')
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [2]:
DATA_DIR = '/kaggle/input/datasets/ffftory/acdc-full/weather_dataset_complete'
BATCH_SIZE = 64
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = 8

print(f"Используется устройство: {DEVICE}")
print(f"Количество классов: {NUM_CLASSES}")

Используется устройство: cuda
Количество классов: 8


In [3]:
class_names = ['clear', 'fog', 'night', 'night_fog', 'night_rain', 'night_snow', 'rain', 'snow']
class_names.sort()
print(f"Классы: {class_names}")

Классы: ['clear', 'fog', 'night', 'night_fog', 'night_rain', 'night_snow', 'rain', 'snow']


In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [5]:
# Загружаем train, val, test
train_dataset = datasets.ImageFolder(
    root=os.path.join(DATA_DIR, 'train'),
    transform=transform
)

val_dataset = datasets.ImageFolder(
    root=os.path.join(DATA_DIR, 'val'),
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root=os.path.join(DATA_DIR, 'test'),
    transform=transform
)


train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    pin_memory=True,
    num_workers=2,
    prefetch_factor=2
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    pin_memory=True,
    num_workers=2,
    prefetch_factor=2
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    pin_memory=True,
    num_workers=2,
    prefetch_factor=2
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Train samples: 4227
Val samples: 1071
Test samples: 3120


In [6]:
print(f"Индексы классов: {train_dataset.class_to_idx}")

Индексы классов: {'clear': 0, 'fog': 1, 'night': 2, 'night_fog': 3, 'night_rain': 4, 'night_snow': 5, 'rain': 6, 'snow': 7}


In [7]:
# Модель
model = models.resnet18(weights="IMAGENET1K_V1")

# Заменяем только последний слой под наши 8 классов
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 185MB/s]


In [8]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for inputs, labels in tqdm(loader, desc="Training"):
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    
    return epoch_loss, epoch_acc

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc="Evaluating"):
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    
    return epoch_loss, epoch_acc, all_preds, all_labels

In [9]:
# Обучение
best_val_acc = 0.0
best_model_state = None

for epoch in range(NUM_EPOCHS):
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    
    # Validation
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        print(f"Новая лучшая модель! Val Acc: {val_acc:.4f}")

print(f"Лучшая точность на валидации: {best_val_acc:.4f}")

Epoch 1/10


Training: 100%|██████████| 67/67 [02:44<00:00,  2.45s/it]


Train Loss: 0.3701, Train Acc: 0.8881


Evaluating: 100%|██████████| 17/17 [00:43<00:00,  2.58s/it]


Val Loss: 0.0826, Val Acc: 0.9739
Новая лучшая модель! Val Acc: 0.9739
Epoch 2/10


...

Train Loss: 0.0039, Train Acc: 1.0000


Evaluating: 100%|██████████| 17/17 [00:38<00:00,  2.27s/it]


Val Loss: 0.0466, Val Acc: 0.9879
Новая лучшая модель! Val Acc: 0.9879
Epoch 7/10


Training:   9%|▉         | 6/67 [00:17<02:54,  2.86s/it]


KeyboardInterrupt: 

In [10]:
# Тестирование 
model.load_state_dict(best_model_state)

test_loss, test_acc, all_preds, all_labels = evaluate(model, test_loader, criterion, DEVICE)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")


precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

print(f"Macro Average Metrics:")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

precision_w = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall_w = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
f1_w = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print(f"Weighted Average Metrics:")
print(f"Precision: {precision_w:.4f}")
print(f"Recall: {recall_w:.4f}")
print(f"F1-Score: {f1_w:.4f}")

Evaluating: 100%|██████████| 49/49 [01:49<00:00,  2.24s/it]

Test Loss: 0.3195
Test Accuracy: 0.9103
Macro Average Metrics:
Precision: 0.9158
Recall: 0.8427
F1-Score: 0.8554
Weighted Average Metrics:
Precision: 0.9206
Recall: 0.9103
F1-Score: 0.9009


In [12]:
target_names = [train_dataset.classes[i] for i in range(len(train_dataset.classes))]

from sklearn.metrics import precision_recall_fscore_support

precision_per_class, recall_per_class, f1_per_class, support = precision_recall_fscore_support(
    all_labels, all_preds, average=None, zero_division=0
)

print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<8}")
for i, class_name in enumerate(target_names):
    print(f"{class_name:<15} {precision_per_class[i]:<12.4f} {recall_per_class[i]:<12.4f} {f1_per_class[i]:<12.4f} {support[i]:<8}")

Class           Precision    Recall       F1-Score     Support 
clear           0.9671       1.0000       0.9833       500     
fog             0.9653       1.0000       0.9823       500     
night           0.7511       0.9960       0.8564       500     
night_fog       0.9006       0.7368       0.8105       209     
night_rain      0.8122       0.7350       0.7717       200     
night_snow      0.9859       0.3318       0.4965       211     
rain            0.9698       0.9620       0.9659       500     
snow            0.9742       0.9800       0.9771       500     
